In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path(".")

files = [
    ("A_7_5g",  ROOT/"ACoS_7_5g"/"ACoS_7_5g_clean.csv"),
    ("A_15g",   ROOT/"ACoS_15g"/"ACoS_15g_clean.csv"),
    ("A_18g",   ROOT/"ACoS_18g"/"ACoS_18g_clean.csv"),
    ("A_22_5g", ROOT/"ACoS_22_5g"/"ACoS_22_5g_clean.csv"),
]

def load(path):
    df = pd.read_csv(path)
    # Arreglar por si quedaron nombres raros:
    df = df.rename(columns={c.lower().strip():c for c in df.columns})
    if not {"nm","A"}.issubset(df.columns):
        # Si venía como x,y entonces lo corregimos
        c0, c1 = df.columns[:2]
        df = df.rename(columns={c0:"nm", c1:"A"})
    df["nm"] = pd.to_numeric(df["nm"], errors="coerce")
    df["A"]  = pd.to_numeric(df["A"], errors="coerce")
    return df.dropna()[["nm","A"]].sort_values("nm").drop_duplicates()

# ---- MATRIZ SIN INTERPOLAR (INTERSECCIÓN EXACTA) ----

mat = load(files[0][1]).rename(columns={"A": files[0][0]})

for label, path in files[1:]:
    df = load(path).rename(columns={"A": label})
    mat = mat.merge(df, on="nm", how="inner")   # SOLO nm que existen en todos

mat = mat.sort_values("nm").reset_index(drop=True)

out = ROOT / "ACoS_matrix.csv"
mat.to_csv(out, index=False)

print("✔ Matriz generada correctamente:")
print(out)
display(mat.head())

✔ Matriz generada correctamente:
ACoS_matrix.csv


,nm,A_7_5g,A_15g,A_18g,A_22_5g
0,235.0,0.726,2.476,1.470,1.369
1,235.5,0.717,2.423,1.450,1.351
2,236.0,0.709,2.370,1.432,1.334
3,236.5,0.701,2.321,1.415,1.318
4,237.0,0.693,2.278,1.399,1.303
